# AutoLearnMeds — Bootstrap

Run **once per Colab session**. After it succeeds, copy the printed `update_ssh_config.sh` line into a terminal on your Mac, then connect via VSCode Remote-SSH to the host alias `autolearnmeds-colab`.

Required: Colab Pro+ runtime, A100 GPU, **Background execution enabled**.

## Why two phases?

`google.colab.auth.authenticate_user()` only works from a notebook cell (it needs the IPython kernel for the consent dialog and to write Application Default Credentials). So this cell does the Colab-API auth in Python first, then shells out to `colab_bootstrap.sh` for the pure-shell parts (gcsfuse, git clone to local SSD, uv sync, SSH tunnel, daemons).

Data is uploaded to `gs://${AUTOLEARNMEDS_GCS_BUCKET}/raw/...` directly by the user. **Drive is not used** — putting the project on Drive triggers Drive's API quota when uv sync writes thousands of small files. The repo is cloned to Colab local SSD (`/content/AutoLearnMeds`) instead.

In [ ]:
# REPLACE these values before running. Keep this notebook private.
import os

# Required
os.environ['AUTOLEARNMEDS_GCS_BUCKET']   = 'gs://auto_learn_meds'
os.environ['AUTOLEARNMEDS_REPO_URL']     = 'https://github.com/n-suman/AutoLearnMeds.git'
os.environ['AUTOLEARNMEDS_SSH_PASSWORD'] = 'cWbDRcUsDzyWkYXRdhno8IzD4yqwTjVl'

# Optional
os.environ['AUTOLEARNMEDS_BRANCH']       = 'phase-0-plumbing'

# === Colab-API step (must run from notebook context) ===
print('[notebook] Authenticating for Google Cloud (gsutil + gcsfuse)...')
from google.colab import auth
auth.authenticate_user()

from google.colab import userdata
os.environ['CLOUDFLARED_TUNNEL_CREDS'] = userdata.get('CLOUDFLARED_TUNNEL_CREDS')

# === Hand off to bash bootstrap for pure-shell steps ===
BRANCH = os.environ['AUTOLEARNMEDS_BRANCH']
REPO   = os.environ['AUTOLEARNMEDS_REPO_URL'].replace('https://github.com/', '').replace('.git', '')
!curl -sSL https://raw.githubusercontent.com/{REPO}/{BRANCH}/scripts/colab_bootstrap.sh -o /tmp/colab_bootstrap.sh
!bash /tmp/colab_bootstrap.sh

## After the bootstrap finishes

1. Read the cloudflared hostname from the output above.
2. On your **Mac**, in your terminal: `cd /Users/apple/AutoLearnMeds && ./scripts/update_ssh_config.sh <hostname>`
3. In **VSCode**: `Cmd-Shift-P` -> Remote-SSH: Connect to Host -> `autolearnmeds-colab`.
4. From inside VSCode, open a terminal and run `make verify` to confirm the rig is green.

If anything fails, the agent (Claude Code in your VSCode) reads `/tmp/keepalive.log`, `/tmp/sync_to_gcs.log`, and `/tmp/colab_ssh_output.txt` to diagnose.

In [ ]:
# Keepalive cell — runs forever to keep the Colab runtime + browser session active.
# Each print() triggers a websocket message kernel -> browser, defeating both the
# kernel-idle timer AND the browser-session-idle detection. Complements the JS
# console heartbeat (which catches a different failure mode: browser tab in BG).
#
# Stop with Runtime > Interrupt execution. Do NOT clear output or close this tab.
import time, datetime
print("[keepalive] starting — this cell runs forever; do not interrupt", flush=True)
i = 0
while True:
    i += 1
    print(f"[keepalive] {datetime.datetime.utcnow().isoformat()} #{i}", flush=True)
    time.sleep(60)
